# 已存完整账户的成本路径分歧
固定三个模型族及全部0/2/5bp成对比较，保留全部944区间。只作描述性分解，不改阈值、不重新拟合。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/opening_cost_path_20261015';parent=root/'research/opening_state_training_20261015';plan=json.loads((card/'plan.json').read_text());s=json.loads((parent/'summary.json').read_text())
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=s['provider_uri'],region='us')
names=[f'analysis/native/{family}_{slip}' for family in plan['families'] for slip in [0,2,5]]
objects,audit=_source(root,plan['source'],names)
rows=[];firsts=[];annual=[];details={}
fields=['execute_open_target','recovery_action','risk_state','risk_clock','current_beta_cap']
for family in plan['families']:
 for lo,hi in plan['pairs']:
  a=objects[f'analysis/native/{family}_{lo}']['authority'];b=objects[f'analysis/native/{family}_{hi}']['authority']
  assert a.trade_date.equals(b.trade_date)
  x=np.log1p(b.strategy_return)-np.log1p(a.strategy_return)
  assert np.isclose(x.sum(),np.log(b.equity.iloc[-1]/a.equity.iloc[-1]),atol=1e-9)
  changes={field:~np.isclose(a[field].astype(float),b[field].astype(float),rtol=0,atol=1e-8,equal_nan=True) for field in fields}
  state=changes['risk_state'];cap=changes['current_beta_cap']
  rows.append({'family':family,'low_bps':lo,'high_bps':hi,'relative_terminal_wealth':b.equity.iloc[-1]/a.equity.iloc[-1],'total_log_gap':x.sum(),'state_different_days':int(state.sum()),'state_different_day_log_gap':x[state].sum(),'cap_different_days':int(cap.sum()),'cap_different_day_log_gap':x[cap].sum(),'open_decision_different_days':int(changes['execute_open_target'].sum()),'broker_fee_delta_usd':b.broker_and_external_fees_usd.sum()-a.broker_and_external_fees_usd.sum(),'slippage_delta_usd':b.slippage_reserve_usd.sum()-a.slippage_reserve_usd.sum()})
  detail=pd.DataFrame({'trade_date':a.trade_date,'log_gap':x,'low_dd':a.portfolio_drawdown,'high_dd':b.portfolio_drawdown,'low_state':a.risk_state,'high_state':b.risk_state,'low_cap':a.current_beta_cap,'high_cap':b.current_beta_cap})
  details[f'{family}_{lo}_{hi}']=detail
  for field,mask in changes.items():
   if not mask.any():continue
   i=int(np.flatnonzero(mask)[0]);firsts.append({'family':family,'low_bps':lo,'high_bps':hi,'field':field,'first_date':str(a.trade_date.iloc[i].date()),'low_value':float(a[field].iloc[i]),'high_value':float(b[field].iloc[i]),'low_drawdown':a.portfolio_drawdown.iloc[i],'high_drawdown':b.portfolio_drawdown.iloc[i]})
  for year,part in detail.groupby(detail.trade_date.dt.year):annual.append({'family':family,'low_bps':lo,'high_bps':hi,'year':year,'log_gap':part.log_gap.sum()})
tables={'path_comparison':pd.DataFrame(rows),'first_divergence':pd.DataFrame(firsts),'annual_log_gap':pd.DataFrame(annual)}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/daily_paths','analysis/tables','inputs/source_audit']}
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='analysis',daily_paths=details,tables=tables);rec.save_objects(artifact_path='inputs',source_audit=audit,plan=plan);rec.log_metrics(reused_native_accounts=9,registered_artifact_count=4)
summary={'experiment':card.name,'decision':'diagnostic_only_no_promotion','new_fits':0,'new_backtests':0,'reused_native_accounts':9,'provider_uri':s['provider_uri'],'region':'us','as_of':s['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':'State-day and cap-day log contributions overlap; descriptive partition, not causal attribution. No fixed-path counterfactual performed.'}
for name,t in tables.items():t.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':s['provider_uri'],'artifact_policy':policy}))
binding=json.loads((parent/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(a.rglob('*')) if p.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(tables['path_comparison'].to_string(index=False))


# 成本路径归因：阈值分叉放大账户差异，不代表策略改善

本轮0拟合、0新增回测，读取已验证Qlib Recorder中候选、配对对照、长期小模型的9个原生完整账户，覆盖各自全部944区间，固定比较0/2、0/5、2/5bp。逐日对数收益差与期末财富比的对数逐组核对一致。所有年度与首次分歧均报告，没有挑收益最好区间。

候选2bp/0bp期末财富比1.18450，对数收益差0.16932；风险状态不同24天，对应对数差0.14124，约占83.4%。仓位上限不同55天，对应0.18101，其他日抵消一部分。状态日和上限日重叠，两项不可相加，也不是阈值的因果贡献。开盘执行决策不同40天，首次2023-08-16；恢复动作没有分歧。

首次仓位上限分歧为2025-06-25：0bp已知回撤21.8946%，2bp为22.1485%，在固定22%紧急线两侧，beta上限分别1.3271与0.3。首次风险状态分歧为2026-04-17：回撤12.2179%与9.2105%，固定12%线使0bp进入停止状态、2bp保持正常，上限0.2与1.3438。后者已包含前期持仓历史的积累差异，不能说只差2bp费用直接造成3个百分点回撤差。

2bp相对0bp的年度对数收益差：2023 -0.002806，2024 -0.004164，2025 +0.044863，2026 +0.131428。新增滑点支出约2038美元，券商/外部费用多24.44美元；较高终值显然不能解释为费用更低。需要固定决策反事实才能分离直接费用和不同持仓带来的影响。

配对对照2bp/0bp财富比0.89851，状态不同24天；长期小模型为0.98384，状态不同43天。长期小模型5bp/0bp财富比又为1.01252，说明路径敏感性并非新开盘字段独有。详见path_comparison.csv、first_divergence.csv和annual_log_gap.csv。

这些结果与当前确定性阈值规则一致，未证明程序错误，也未证明调整阈值可带来独立超额。既有stop_rearm_20261014已测试固定重新武装机制且未通过，不重开等待时长、恢复阈值或重新武装搜索。上一轮开盘特征候选仍不晋升，生产不变；稳定净SOXX超额未完成。

下一信息需求：在同一来源和同一固定模型下，保留一条预先指定的执行目标轨迹，仅重结算不同费用，作为诊断反事实，与完整独立账户的费用/持仓影响区分。该轨迹不能替代真实独立风险账户，也不能作为晋升证据；未完成此反事实前不声称已确定因果修复。证据通过NAS逐对象冷恢复验证。
